# Non-linearity, Hetero-skedasticity, and Normality Through Data

In this section we will be looking at non-linearity, hetero-skedasticity and normality using the Ford Focus dataset.

We start by extracting the data and running linear regression on it. Run the code below to fit the model. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm

In [ ]:
data = pd.read_csv("data/Focus.csv")

# Extracting variables for regression
prices = data["price"]
mileages = data["mileage"]
engine_sizes = data["engineSize"]
X_vals = data[["mileage", "engineSize"]]

# Set up observation vector and design matrix
Y = prices
X = sm.add_constant(X_vals)

# Fit the OLS model
model = sm.OLS(Y, X)
results = model.fit()

# Return the summary of the regression results
print(results.summary())

estimated_values = results.predict(X)

Note the error in the results summary: `The condition number is large, 3.98e+05. This might indicate that there are
strong multicollinearity or other numerical problems.` We can plot our data to investigate further. 

In [ ]:
plt.figure(figsize=(8, 10))
plt.subplot(3, 1, 1)
x = np.linspace(np.min(estimated_values), np.max(estimated_values), 100)
plt.scatter(prices, estimated_values, label="Data Points", color="blue")
plt.plot(x, 1 * x, color="red", label="Regression Line")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Linear Regression: Expected vs Actual Prices")
plt.legend()
plt.grid()

plt.subplot(3, 1, 2)
plt.scatter(mileages, prices, label="Data Points", color="blue")
plt.xlabel("Mileage")
plt.ylabel("Price")
plt.title("Linear Regression: Prices vs Mileage")
plt.legend()
plt.grid()

plt.subplot(3, 1, 3)
plt.scatter(engine_sizes, prices, label="Data Points", color="blue")
plt.xlabel("Engine Size")
plt.ylabel("Price")
plt.title("Linear Regression: Prices vs Engine Sizes")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

We observe from these plots that there is hetero-skedasticity in the data. We can solve this by applying a logarithmic transformation to the price values. Run the code below to execute this transformation and observe the resulting model.

In [ ]:
# Set up observation vector and design matrix
Y = np.log(prices)
X = sm.add_constant(X_vals)

# Fit the OLS model
model = sm.OLS(Y, X)
results = model.fit()
estimated_values = results.predict(X)


# Return the summary of the regression results
print(results.summary())

x = np.linspace(np.min(estimated_values), np.max(estimated_values), 100)
plt.figure(figsize=(8, 10))
plt.subplot(3, 1, 1)
plt.scatter(Y, estimated_values, label="Data Points", color="blue")
plt.plot(x, 1 * x, color="red", label="Regression Line")
plt.xlabel("Actual Price")
plt.ylabel("Estimated Price")
plt.title("Linear Regression: Expected vs Actual Prices")
plt.legend()
plt.grid()

plt.subplot(3, 1, 2)
plt.scatter(mileages, Y, label="Data Points", color="blue")
plt.xlabel("Mileage")
plt.ylabel("Price")
plt.title("Linear Regression: Prices vs Mileage")
plt.legend()
plt.grid()

plt.subplot(3, 1, 3)
plt.scatter(engine_sizes, Y, label="Data Points", color="blue")
plt.xlabel("Engine Size")
plt.ylabel("Price")
plt.title("Linear Regression: Prices vs Engine Size")
plt.legend()
plt.grid()

plt.tight_layout()

plt.show()

Even with this transformation we still see the error message. This suggests there may be some non-linearity. We can confirm this through residual analysis. Observe the normality of the residuals.

In [ ]:
residuals = results.resid

plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.scatter(mileages, residuals, alpha=0.6, color="blue")
plt.axhline(y=0, color="red", linestyle="--", linewidth=2)
plt.xlabel("X Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values")
plt.grid()

plt.subplot(2, 1, 2)
plt.hist(residuals, bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Residuals")
plt.show()

To remedy this non-linearity we can transform engine size to be expressed in dm rather than $dm^3=l$ by applying a cube root to this value.

In [ ]:
# Set up observation vector and design matrix
Y = np.log(prices)
X_matrix = np.column_stack([mileages, engine_sizes, np.cbrt(engine_sizes)])
X = sm.add_constant(X_matrix)

# Fit the OLS model
model = sm.OLS(Y, X)
results = model.fit()
estimated_values = results.predict(X)


# Return the summary of the regression results
print(results.summary())

# Plot Actual vs Predicted Prices
plt.figure(figsize=(10, 6))
plt.scatter(Y, estimated_values, label="Data Points", color="blue")
x = np.linspace(np.min(estimated_values), np.max(estimated_values), 100)
plt.plot(x, 1 * x, color="red", label="Regression Line")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Linear Regression: Actual vs Predicted Price")
plt.legend()
plt.grid()
plt.show()

# Plot Prices vs Engine Size
plt.figure(figsize=(10, 6))
plt.scatter(engine_sizes, Y, label="Data Points", color="blue")
plt.xlabel("Engine Size")
plt.ylabel("Price")
plt.title("Linear Regression: Prices vs Engine Size")
plt.legend()
plt.grid()
plt.show()

# plot residuals
residuals = results.resid

plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.scatter(mileages, residuals, alpha=0.6, color="blue")
plt.axhline(y=0, color="red", linestyle="--", linewidth=2)
plt.xlabel("X Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values")
plt.grid()

plt.subplot(2, 1, 2)
plt.hist(residuals, bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Residuals")
plt.show()